# MobilePlantViT Block Validation Notebook

This notebook provides visual verification for all neural network blocks.

## Contents
1. Setup and Imports
2. Block-by-Block Testing
3. Full Pipeline Verification
4. Benchmark Results

In [ ]:
# Setup and Imports
import sys
sys.path.insert(0, '..')  # Add parent directory to path

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

# Set random seed for reproducibility
torch.manual_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Utility Imports

Import testing and benchmarking utilities.

In [ ]:
# Import utilities
from src.utils.testing import (
    check_output_shape,
    check_gradient_flow,
    check_no_nan_inf,
    count_parameters,
)

from src.utils.benchmarking import (
    time_forward_backward,
    warmup_and_benchmark,
)

print("Utilities imported successfully!")

## 2. Block Imports

These imports will be uncommented as blocks are implemented.

In [ ]:
# Block imports - uncomment as implemented
# from src.blocks import (
#     GhostConv,
#     CoordAtt,
#     FusedInvertedResidualBlock,
#     PatchEmbedding,
#     PositionalEncoding,
#     LinearDifferentialAttention,
#     BottleneckFFN,
#     ResidualLayerNormBlock,
#     GlobalAveragePooling,
#     ClassifierHead,
# )

print("Block imports ready (uncomment as implemented)")

In [ ]:
# GhostConv Validation
print("=" * 60)
print("GhostConv Validation")
print("=" * 60)

from src.blocks import GhostConv

# Test configurations
configs = [
    {"name": "Basic (3→64)", "params": {"inp": 3, "oup": 64}, "input_shape": (2, 3, 224, 224)},
    {"name": "With stride=2", "params": {"inp": 64, "oup": 128, "stride": 2}, "input_shape": (2, 64, 56, 56)},
    {"name": "Ratio=4", "params": {"inp": 64, "oup": 64, "ratio": 4}, "input_shape": (2, 64, 56, 56)},
]

for cfg in configs:
    ghost = GhostConv(**cfg['params'])
    x = torch.randn(*cfg['input_shape'])
    
    success = validate_block(
        ghost, 
        cfg['input_shape'], 
        None,  # Will check actual output
        cfg['name']
    )
    
    # Show internal channels
    print(f"  Internal: init_channels={ghost.init_channels}, new_channels={ghost.new_channels}")

print("\n✅ GhostConv validation complete!")

In [ ]:
# CoordAtt Validation
print("=" * 60)
print("Coordinate Attention Validation")
print("=" * 60)

from src.blocks import CoordAtt, HSigmoid, HSwish

# Test configurations
configs = [
    {"name": "Basic (64→64)", "params": {"inp": 64, "oup": 64}, "input_shape": (2, 64, 56, 56)},
    {"name": "With reduction=16", "params": {"inp": 128, "oup": 128, "reduction": 16}, "input_shape": (2, 128, 28, 28)},
    {"name": "Rectangular input", "params": {"inp": 64, "oup": 64}, "input_shape": (2, 64, 56, 28)},
]

for cfg in configs:
    coord_att = CoordAtt(**cfg['params'])
    x = torch.randn(*cfg['input_shape'])
    
    # Get expected output shape (same as input for CoordAtt)
    expected_shape = cfg['input_shape']
    
    success = validate_block(
        coord_att, 
        cfg['input_shape'], 
        expected_shape,
        cfg['name']
    )
    
    # Show internal mip value
    print(f"  Internal: mip={coord_att.mip}")
    
    # Visualize attention maps
    coord_att.eval()
    with torch.no_grad():
        a_h, a_w = coord_att.get_attention_maps(x)
    print(f"  Attention shapes: a_h={a_h.shape}, a_w={a_w.shape}")

print("\n✅ Coordinate Attention validation complete!")

## 3. Template: Block Validation

Use this template for validating each block.

In [ ]:
def validate_block(block, input_shape, expected_output_shape, block_name):
    """
    Validate a block's functionality.
    
    Args:
        block: The neural network block to validate
        input_shape: Tuple of input dimensions
        expected_output_shape: Tuple of expected output dimensions
        block_name: Name of the block for display
    """
    print(f"\n{'='*60}")
    print(f"Validating: {block_name}")
    print(f"{'='*60}")
    
    # Create random input
    x = torch.randn(*input_shape)
    print(f"Input shape: {x.shape}")
    
    # Shape test
    try:
        check_output_shape(block, x, expected_output_shape)
        print(f"✅ Output shape correct: {expected_output_shape}")
    except AssertionError as e:
        print(f"❌ Shape error: {e}")
        return False
    
    # Gradient flow test
    grad_info = check_gradient_flow(block, x)
    all_grads = all(grad_info.values())
    if all_grads:
        print(f"✅ Gradients flow to all {len(grad_info)} parameters")
    else:
        missing = [k for k, v in grad_info.items() if not v]
        print(f"❌ Missing gradients for: {missing}")
    
    # NaN/Inf check
    block.eval()
    with torch.no_grad():
        output = block(x)
    try:
        check_no_nan_inf(output, "output")
        print(f"✅ No NaN/Inf in output")
    except AssertionError as e:
        print(f"❌ {e}")
        return False
    
    # Parameter count
    params = count_parameters(block)
    print(f"📊 Parameters: {params['total']:,} total, {params['trainable']:,} trainable")
    
    return True

# Example usage (will work once blocks are implemented):
# validate_block(GhostConv(3, 64), (2, 3, 224, 224), (2, 64, 224, 224), "GhostConv")

## 4. Visualization Helpers

In [ ]:
def visualize_feature_maps(feature_map, title="Feature Maps", num_channels=8):
    """
    Visualize feature maps from a convolutional layer.
    
    Args:
        feature_map: Tensor of shape (B, C, H, W)
        title: Title for the plot
        num_channels: Number of channels to display
    """
    if feature_map.dim() != 4:
        print("Expected 4D tensor (B, C, H, W)")
        return
    
    # Take first batch
    fm = feature_map[0].detach().cpu().numpy()
    
    # Limit channels
    num_channels = min(num_channels, fm.shape[0])
    
    fig, axes = plt.subplots(2, num_channels // 2, figsize=(12, 6))
    axes = axes.flatten()
    
    for i in range(num_channels):
        axes[i].imshow(fm[i], cmap='viridis')
        axes[i].set_title(f'Ch {i}')
        axes[i].axis('off')
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

def visualize_attention_map(attention, title="Attention Map"):
    """
    Visualize attention weights.
    
    Args:
        attention: Tensor of shape (B, heads, N, N) or (B, N, N)
        title: Title for the plot
    """
    attn = attention.detach().cpu().numpy()
    
    if attn.ndim == 4:
        # Average over heads
        attn = attn.mean(axis=1)
    
    # Take first batch
    attn = attn[0]
    
    plt.figure(figsize=(8, 8))
    plt.imshow(attn, cmap='hot')
    plt.colorbar()
    plt.title(title)
    plt.xlabel("Key Position")
    plt.ylabel("Query Position")
    plt.show()

print("Visualization helpers defined!")

## 5. Benchmark Results Template

In [ ]:
def run_benchmarks(blocks_config):
    """
    Run benchmarks for multiple blocks.
    
    Args:
        blocks_config: List of tuples (block, input_shape, name)
    
    Returns:
        Dictionary of benchmark results
    """
    results = {}
    
    for block, input_shape, name in blocks_config:
        print(f"\nBenchmarking: {name}")
        
        x = torch.randn(*input_shape)
        
        timing = warmup_and_benchmark(
            block, x, 
            num_runs=50, 
            warmup_runs=10, 
            device='cpu',  # Change to 'cuda' if available
            include_backward=True
        )
        
        results[name] = {
            'forward_ms': timing['forward_only_mean_ms'],
            'backward_ms': timing.get('backward_mean_ms', 0),
            'throughput': timing['throughput_samples_per_sec'],
            'params': count_parameters(block)['total']
        }
        
        print(f"  Forward: {timing['forward_only_mean_ms']:.3f} ms")
        print(f"  Backward: {timing.get('backward_mean_ms', 0):.3f} ms")
        print(f"  Throughput: {timing['throughput_samples_per_sec']:.1f} samples/sec")
    
    return results

# Will be used after blocks are implemented:
# benchmark_results = run_benchmarks([
#     (GhostConv(3, 64), (2, 3, 224, 224), "GhostConv"),
#     (FusedInvertedResidualBlock(64, 64), (2, 64, 56, 56), "FusedIR"),
#     # ... more blocks
# ])

print("Benchmark function defined!")